In [1]:
# نصب پکیج‌های مورد نیاز
!pip install -q -U transformers datasets accelerate evaluate jiwer
!pip install -q -U soundfile librosa mutagen
!pip install -q -U datacollective

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 85.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 80.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 

In [2]:
import os
import glob
import random
import warnings

import numpy as np
import pandas as pd
import torch
import soundfile as sf
from mutagen.mp3 import MP3

from datasets import Dataset, Audio, load_dataset
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate

warnings.filterwarnings("ignore")

# ---------------- تنظیمات کلی ----------------
MODEL_NAME = "openai/whisper-small"
LANGUAGE = "persian"
TASK = "transcribe"
SAMPLE_RATE = 16000
SEED = 42

TARGET_CV_HOURS = 20
TARGET_YT_HOURS = 0

CV_EXTRACT_DIR = "/tmp/common_voice_fa_extracted"
YT_CACHE_DIR = "/tmp/yt_audio_cache"
OUTPUT_DIR = "/kaggle/working/whisper-small-fa-finetuned"

random.seed(SEED)
np.random.seed(SEED)

n_gpu = torch.cuda.device_count()
print("تعداد GPU شناسایی‌شده:", n_gpu)
for i in range(n_gpu):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
if n_gpu > 1:
    print("Seq2SeqTrainer به‌صورت خودکار از هر", n_gpu, "تا GPU با DataParallel استفاده می‌کنه، بدون نیاز به کد اضافه.")

تعداد GPU شناسایی‌شده: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4
Seq2SeqTrainer به‌صورت خودکار از هر 2 تا GPU با DataParallel استفاده می‌کنه، بدون نیاز به کد اضافه.


In [3]:
try:
    from kaggle_secrets import UserSecretsClient
    MDC_API_KEY = UserSecretsClient().get_secret("MDC_API_KEY")
except Exception:
    MDC_API_KEY = "خودتان جایگذاری کنید"

os.environ["MDC_API_KEY"] = MDC_API_KEY

CV_DATASET_ID = "cmqinhw5100v8nr07gyg5gi4v"  # Common Voice Scripted Speech - Persian

from datacollective import download_dataset
import tarfile

cv_archive_path = str(download_dataset(CV_DATASET_ID))
print("مسیر آرشیو دانلود‌شده:", cv_archive_path)

if os.path.isfile(cv_archive_path):
    if not os.path.isdir(CV_EXTRACT_DIR) or not os.listdir(CV_EXTRACT_DIR):
        os.makedirs(CV_EXTRACT_DIR, exist_ok=True)
        print("در حال extract کردن آرشیو... (ممکنه چند دقیقه طول بکشه)")
        with tarfile.open(cv_archive_path, "r:gz") as tar:
            tar.extractall(path=CV_EXTRACT_DIR)
        print("extract تموم شد.")
    else:
        print("قبلاً extract شده، از کش استفاده می‌کنیم.")
    cv_root = CV_EXTRACT_DIR
elif os.path.isdir(cv_archive_path):
    cv_root = cv_archive_path
else:
    raise RuntimeError(f"مسیر برگشتی نه فایله نه پوشه: {cv_archive_path}")

print("مسیر نهایی برای جستجوی tsv/mp3:", cv_root)
for root, dirs, files in os.walk(cv_root):
    depth = root[len(cv_root):].count(os.sep)
    if depth > 2:
        dirs[:] = []
        continue
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root)}/")
    for f in sorted(files)[:5]:
        print(f"{indent}  {f}")
    if len(files) > 5:
        print(f"{indent}  ... و {len(files) - 5} فایل دیگر")

█████████████████████████████████████████████████🦊 100.0% (10.5 GB/10.5 GB) Average: 81.0 MB/s Total time: 02:12
مسیر آرشیو دانلود‌شده: /root/.mozdata/datasets/common-voice-scripted-speech-26-0-persia-65a9441e.tar.gz
در حال extract کردن آرشیو... (ممکنه چند دقیقه طول بکشه)
extract تموم شد.
مسیر نهایی برای جستجوی tsv/mp3: /tmp/common_voice_fa_extracted
common_voice_fa_extracted/
  cv-corpus-26.0-2026-06-12/
    fa/
      README.md
      clip_durations.tsv
      dev.tsv
      invalidated.tsv
      other.tsv
      ... و 6 فایل دیگر


In [4]:
import csv

tsv_candidates = (
    glob.glob(os.path.join(cv_root, "**", "validated.tsv"), recursive=True)
    or glob.glob(os.path.join(cv_root, "**", "train.tsv"), recursive=True)
    or glob.glob(os.path.join(cv_root, "**", "*.tsv"), recursive=True)
)
assert tsv_candidates, "هیچ فایل .tsv پیدا نشد — ساختار پوشه چاپ‌شده در سلول قبل رو دستی چک کن."
cv_tsv_path = tsv_candidates[0]
print("فایل متادیتای انتخاب‌شده:", cv_tsv_path)

mp3_candidates = glob.glob(os.path.join(cv_root, "**", "*.mp3"), recursive=True)
assert mp3_candidates, "هیچ فایل mp3 پیدا نشد."
clips_dir = os.path.dirname(mp3_candidates[0])
print("پوشه کلیپ‌ها:", clips_dir)

# quoting=csv.QUOTE_NONE لازمه چون tsv کامان‌ویس از quote-escaping سبک CSV استفاده نمی‌کنه
cv_df = pd.read_csv(cv_tsv_path, sep="\t", quoting=csv.QUOTE_NONE)
print("ستون‌های موجود:", cv_df.columns.tolist())
assert "path" in cv_df.columns and "sentence" in cv_df.columns, (
    f"ستون‌های 'path'/'sentence' پیدا نشد. ستون‌های موجود: {cv_df.columns.tolist()}"
)

cv_df["full_path"] = cv_df["path"].apply(lambda p: os.path.join(clips_dir, p))
cv_df = cv_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)


def mp3_duration_sec(path):
    try:
        return MP3(path).info.length
    except Exception:
        return None


selected = []
total_sec = 0.0
target_sec = TARGET_CV_HOURS * 3600

for _, row in cv_df.iterrows():
    dur = mp3_duration_sec(row["full_path"])
    if dur is None:
        continue
    sentence = str(row["sentence"])
    if len(sentence) > 400:  # فیلتر ایمنی جمله‌های غیرعادی طولانی (نشونه‌ی خرابی parsing)
        continue
    selected.append({"path": row["full_path"], "sentence": sentence, "source": "common_voice"})
    total_sec += dur
    if total_sec >= target_sec:
        break

print(f"Common Voice: {len(selected)} کلیپ، {total_sec/3600:.2f} ساعت")
cv_records = selected

فایل متادیتای انتخاب‌شده: /tmp/common_voice_fa_extracted/cv-corpus-26.0-2026-06-12/fa/validated.tsv
پوشه کلیپ‌ها: /tmp/common_voice_fa_extracted/cv-corpus-26.0-2026-06-12/fa/clips
ستون‌های موجود: ['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment']
Common Voice: 18327 کلیپ، 20.00 ساعت


In [5]:
all_records = cv_records
random.shuffle(all_records)

full_df = pd.DataFrame(all_records)
print("مجموع کلیپ‌ها:", len(full_df))
print(full_df["source"].value_counts())

full_ds = Dataset.from_pandas(full_df.reset_index(drop=True))
full_ds = full_ds.rename_column("path", "audio")
full_ds = full_ds.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))

split_ds = full_ds.train_test_split(test_size=0.1, seed=SEED)
train_ds = split_ds["train"]
eval_ds = split_ds["test"]

print("Train:", len(train_ds), " | Eval:", len(eval_ds))

مجموع کلیپ‌ها: 18327
source
common_voice    18327
Name: count, dtype: int64
Train: 16494  | Eval: 1833


In [6]:
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

In [7]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch


# num_proc عمداً حذف شده تا خطای احتمالی توی پردازش اصلی (نه subprocess) دیده بشه
train_ds = train_ds.map(prepare_dataset, remove_columns=train_ds.column_names)
eval_ds = eval_ds.map(prepare_dataset, remove_columns=eval_ds.column_names)

# فیلتر ایمنی طول لیبل: علاوه بر جلوگیری از خطای indexing، از batch های سنگین/OOM هم جلوگیری می‌کنه
train_ds = train_ds.filter(lambda b: len(b["labels"]) <= 225)
eval_ds = eval_ds.filter(lambda b: len(b["labels"]) <= 225)
print("بعد از فیلتر -> Train:", len(train_ds), " | Eval:", len(eval_ds))

Map:   0%|          | 0/16494 [00:00<?, ? examples/s]

Map:   0%|          | 0/1833 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16494 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1833 [00:00<?, ? examples/s]

بعد از فیلتر -> Train: 16494  | Eval: 1833


In [13]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [14]:
metric = evaluate.load("wer")


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [15]:
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.generation_config.language = LANGUAGE
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [16]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,       
    gradient_accumulation_steps=2,       # effective batch size = 16 در هر GPU
    per_device_eval_batch_size=4,
    gradient_checkpointing=True,          # حاشیه امن بیشتر برای مدل بزرگ‌تر
    learning_rate=1e-5,
    warmup_steps=100,
    num_train_epochs=3,
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=200,
    eval_steps=200,
    logging_steps=25,
    save_total_limit=2,
    report_to=["none"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,   # نسخه‌های قدیمی‌تر transformers: به‌جاش tokenizer=processor.feature_extractor
)

print("n_gpu که Trainer می‌بینه:", trainer.args.n_gpu, "| batch مؤثر کل =",
      training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps * max(trainer.args.n_gpu, 1))

n_gpu که Trainer می‌بینه: 2 | batch مؤثر کل = 32


In [17]:
trainer.train()

Step,Training Loss,Validation Loss,Wer
200,1.981608,0.494400,51.883122
400,1.532783,0.388983,41.792898
600,1.027007,0.352750,39.119278
800,0.973105,0.331824,37.438954
1000,0.882361,0.311789,35.708964
1200,0.555902,0.311071,34.334906
1400,0.518317,0.306577,33.879646
1548,0.520691,0.303661,33.407830


[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The cu

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=1548, training_loss=1.471746653549431, metrics={'train_runtime': 27230.0651, 'train_samples_per_second': 1.817, 'train_steps_per_second': 0.057, 'total_flos': 1.427978277863424e+19, 'train_loss': 1.471746653549431, 'epoch': 3.0})

In [18]:
save_dir = OUTPUT_DIR + "/final"
trainer.save_model(save_dir)
processor.save_pretrained(save_dir)
print("مدل ذخیره شد در:", save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

مدل ذخیره شد در: /kaggle/working/whisper-small-fa-finetuned/final


In [170]:
model.eval()
idx = random.randint(0, len(eval_ds) - 1)
sample = eval_ds[idx]

input_features = torch.tensor(sample["input_features"]).unsqueeze(0)
if torch.cuda.is_available():
    input_features = input_features.to(model.device)

with torch.no_grad():
    predicted_ids = model.generate(input_features)  # language/task از generation_config خونده می‌شه

predicted_text = processor.batch_decode(predicted_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

label_ids = [l if l != -100 else tokenizer.pad_token_id for l in sample["labels"]]
actual_text = processor.batch_decode([label_ids], skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

print("متن واقعی:      ", actual_text)
print("متن پیش‌بینی‌شده:", predicted_text)

متن واقعی:       تجزیه و تحلیل دقیق و کامل
متن پیش‌بینی‌شده: تزیی و تحلیل دقیق و کامل


## بعدش چیکار کنم؟

- اگه با وجود دو-GPU بازم OOM دیدی، اول `per_device_train_batch_size` رو به ۴ کاهش بده و `gradient_accumulation_steps` رو به ۴ افزایش بده (batch مؤثر همون می‌مونه).
- Kaggle session تا ۱۲ ساعت و سهمیه هفتگی GPU حدود ۳۰ ساعته. چک‌پوینت‌ها هر ۲۰۰ step توی `OUTPUT_DIR` ذخیره می‌شن؛ با `trainer.train(resume_from_checkpoint=True)` می‌تونی از سشن قطع‌شده ادامه بدی.
- اگه نتیجه این تست کوچیک روی `small` منطقی بود، مرحله بعد افزایش `TARGET_CV_HOURS`/`TARGET_YT_HOURS` به سمت دیتاست کامل‌تره.

In [38]:
with open("/kaggle/working/used_clips.txt", "w") as f:
    for r in cv_records:
        f.write(r["path"] + "\n")

In [41]:
from huggingface_hub import login, upload_file

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = "خودتان جایگذاری کنید"

login(token=HF_TOKEN)

REPO_ID = "amirsz8203/whisper-small-fa-finetuned"
ROUND = 1
HOURS_THIS_ROUND = 20


trainer.args.hub_model_id = REPO_ID

trainer.push_to_hub(
    commit_message=f"round {ROUND}: fine-tuned on +{HOURS_THIS_ROUND}h Persian Common Voice",
)
processor.push_to_hub(REPO_ID)

upload_file(
    path_or_fileobj="/kaggle/working/used_clips.txt",
    path_in_repo="used_clips.txt",
    repo_id=REPO_ID,
    repo_type="model",
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/amirsz8203/whisper-small-fa-finetuned/commit/0df3977c76e219cf2c0a88d4477e1ef0d58ecfb6', commit_message='Upload used_clips.txt with huggingface_hub', commit_description='', oid='0df3977c76e219cf2c0a88d4477e1ef0d58ecfb6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/amirsz8203/whisper-small-fa-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='amirsz8203/whisper-small-fa-finetuned'), pr_revision=None, pr_num=None)